# Union de las bases datos y ajustes

En archivo se construira la serie de tiempo de la producción agricola de Colombia, 
usando los datos del Banco de la República, y se construiran las variables dummy para cada reforma agraria.
Cada dataset tiene bases para los datos diferentes, es necario unificarlos para poder hacer un buen analisis
comparativo, ademas se debe ajustar la inflación y aplicar una metodologia de desestacionalizacion para poder hacer
un analisis econometrico de series de tiempo.

In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# para leer los datos en xlsx
import seaborn as sns

In [35]:
# vamos a leer todas las bases de datos

for i in range(3, 9):
    # Creamos la variable globalmente con el nombre dinámico
    globals()[f'df{i}'] = pd.read_excel(f'File ({i}).xlsx', sheet_name='Series de datos', skiprows=1)
    print(f'Variable df{i} creada exitosamente.')



Variable df3 creada exitosamente.
Variable df4 creada exitosamente.
Variable df5 creada exitosamente.
Variable df6 creada exitosamente.
Variable df7 creada exitosamente.
Variable df8 creada exitosamente.


Ya teniendo todas las bases cargadas vamos a limpiar las filas de NA y seleccionar las filas que se llame 1.01.01. Agricultura, ganadería, caza, silvicultura y pesca y 1.01. Agropecuario

In [ ]:
# Creamos una lista con los sectores de interés
sectores_interes = ['1.01.01. Agropecuario, silvicultura, caza y pesca', '1.01. Agropecuario', '1.01.01. Agricultura, silvicultura, caza y pesca'
, '1.01.01. Agricultura, ganadería, caza, silvicultura\ny pesca' ]

for i in range(3, 9):
    # 1. Traemos el dataframe original usando su nombre de variable
    df_actual = globals()[f'df{i}']
    
    # 2. Limpiamos y filtramos el dataframe
    df_actual.dropna(inplace=True)
    filtro_agri = df_actual[df_actual['Unnamed: 0'].isin(sectores_interes)].copy()
    
    # 3. Creamos la nueva variable 
    globals()[f'df{i}_agri'] = filtro_agri
    
    print(f'Creada exitosamente la variable: df{i}_agri con {len(filtro_agri)} filas.')

Creada exitosamente la variable: df3_agri con 1 filas.
Creada exitosamente la variable: df4_agri con 1 filas.
Creada exitosamente la variable: df5_agri con 1 filas.
Creada exitosamente la variable: df6_agri con 1 filas.
Creada exitosamente la variable: df7_agri con 1 filas.
Creada exitosamente la variable: df8_agri con 1 filas.


Ya con las bases donde seleccionamos la agricultura ahora vamos a cambiar los formato de ancho a largo, teniendo en cuenta que las columnas son fechas

In [ ]:
sectores_interes = [
    '1.01.01. Agropecuario, silvicultura, caza y pesca', 
    '1.01. Agropecuario', 
    '1.01.01. Agricultura, silvicultura, caza y pesca',
    '1.01.01. Agricultura, ganadería, caza, silvicultura\ny pesca'
]

for i in range(3, 9):
    # 1. Traer el DF original
    df_actual = globals()[f'df{i}'].copy()
    
    # 2. Limpiar nulos
    df_actual.dropna(inplace=True)
    
    # 3. Filtrar usando tu lista de nombres variables
    df_filtrado = df_actual[df_actual['Unnamed: 0'].isin(sectores_interes)].copy()
    
    # 4. HOMOLOGACIÓN: Forzamos a que todos se llamen igual
    # Así, sin importar cómo venía en el Excel, ahora será estándar
    df_filtrado['Unnamed: 0'] = 'Sector Agropecuario'
    
    # 5. Transformar a formato largo (Melt)
    df_long = pd.melt(
        df_filtrado, 
        id_vars=['Unnamed: 0'], 
        var_name='Fecha', 
        value_name='Valor'
    )
    
    # 6. Limpieza final de la columna Fecha y Renombrar
    df_long.rename(columns={'Unnamed: 0': 'Sector'}, inplace=True)
    df_long['Fecha'] = pd.to_datetime(df_long['Fecha'], dayfirst=True, errors='coerce')
    
    # 7. Guardar en variable global
    globals()[f'df{i}_long'] = df_long
    
    print(f'df{i}_long creado y estandarizado con éxito.')

df3_long creado y estandarizado con éxito.
df4_long creado y estandarizado con éxito.
df5_long creado y estandarizado con éxito.
df6_long creado y estandarizado con éxito.
df7_long creado y estandarizado con éxito.
df8_long creado y estandarizado con éxito.


Con los df creados vamos mirar los años en los cuales inicia y finaliza cada df

In [39]:
for i in range(3, 9):
    df_actual = globals()[f'df{i}_long']
    print(f'Para df{i}_long: Fecha mínima = {df_actual["Fecha"].min()} | Fecha máxima = {df_actual["Fecha"].max()}')

Para df3_long: Fecha mínima = 1925-12-31 00:00:00 | Fecha máxima = 1949-12-31 00:00:00
Para df4_long: Fecha mínima = 1950-12-31 00:00:00 | Fecha máxima = 1970-12-31 00:00:00
Para df5_long: Fecha mínima = 1970-12-31 00:00:00 | Fecha máxima = 1996-12-31 00:00:00
Para df6_long: Fecha mínima = 1994-03-31 00:00:00 | Fecha máxima = 2007-12-31 00:00:00
Para df7_long: Fecha mínima = 2000-03-31 00:00:00 | Fecha máxima = 2017-12-31 00:00:00
Para df8_long: Fecha mínima = 2005-03-31 00:00:00 | Fecha máxima = 2025-12-31 00:00:00


Vemos como del df6 al df8 las series son trimestrales, entonces vamos a pasar estos dos series a anuales para poder hacer un analisis homogeneo de estos 100 años

In [40]:
# Lista de los dataframes que son trimestrales
dfs_trimestrales = [6, 7, 8]

for i in dfs_trimestrales:
    df_temp = globals()[f'df{i}_long'].copy()
    
    # 1. Ponemos la fecha como índice (necesario para resample)
    df_temp.set_index('Fecha', inplace=True)
    
    # 2. Agrupamos por año ('YE' o 'Y') y sumamos los valores
    # Usamos .resample('YE').sum() para sumar los 4 trimestres
    df_anual = df_temp.resample('YE').sum(numeric_only=True).reset_index()
    
    # 3. Re-agregamos la columna de Sector (que se pierde al sumar)
    df_anual['Sector'] = 'Sector Agropecuario'
    
    # 4. Guardamos la nueva versión anual
    globals()[f'df{i}_anual'] = df_anual
    
    print(f'df{i} convertido a anual. Periodos: {len(df_anual)}')

df6 convertido a anual. Periodos: 14
df7 convertido a anual. Periodos: 18
df8 convertido a anual. Periodos: 21


# ANTES DE LA UNION
vamos a trasformar todas las series por medio de un proceso de empalme de cada una de ellas con las siguiente ecuación

$$
Y_{t-1} = \frac{Y_t}{1+G_t}
$$

Donde:
$Y_t$ es es el nivel de la serie nueva
$Y_i$ es es el nivel de la serie vieja
$G_t$ crecimiento de la serie antigua

Entonces recordamos en cual base esta cada uno de los df

df_3 = base 1950
df_4 = base 1958
df_5 = base 1975
df_6 = base 1994
df_7 = base 2005
df_8 = base 2015

Vamos a pasar todas las series a base de 2015

In [48]:
# --- 1. PROCESAMIENTO INICIAL ---
sectores_interes = [
    '1.01.01. Agropecuario, silvicultura, caza y pesca', 
    '1.01. Agropecuario', 
    '1.01.01. Agricultura, silvicultura, caza y pesca',
    '1.01.01. Agricultura, ganadería, caza, silvicultura\ny pesca'
]

for i in range(3, 9):
    df = pd.read_excel(f'File ({i}).xlsx', sheet_name='Series de datos', skiprows=1)
    df.dropna(subset=['Unnamed: 0'], inplace=True)
    df_f = df[df['Unnamed: 0'].isin(sectores_interes)].copy()
    df_f['Unnamed: 0'] = 'Sector Agropecuario'
    
    # Melt y Fechas
    df_l = pd.melt(df_f, id_vars=['Unnamed: 0'], var_name='Fecha', value_name='Valor')
    df_l.rename(columns={'Unnamed: 0': 'Sector'}, inplace=True)
    df_l['Fecha'] = pd.to_datetime(df_l['Fecha'], dayfirst=True, errors='coerce')
    df_l['Año'] = df_l['Fecha'].dt.year
    
    if i in [3, 4, 5]:
        globals()[f'df{i}_long'] = df_l.sort_values('Año')
    else:
        # Anualización para 6, 7 y 8 (Suma de trimestres)
        df_a = df_l.resample('YE', on='Fecha').sum(numeric_only=True).reset_index()
        df_a['Año'] = df_a['Fecha'].dt.year
        df_a['Sector'] = 'Sector Agropecuario'
        globals()[f'df{i}_anual'] = df_a.sort_values('Año')

# --- 2. ESTRATEGIA DE EMPALME EN CASCADA ---
series_ajustadas = {8: df8_anual.copy()}
nombres_map = {7: 'df7_anual', 6: 'df6_anual', 5: 'df5_long', 4: 'df4_long', 3: 'df3_long'}

# Empalme de 7 a 4 (Donde hay años comunes directos)
for i in range(7, 3, -1):
    df_ref = series_ajustadas[i+1]
    df_viej = globals()[nombres_map[i]].copy()
    
    comunes = set(df_ref['Año']) & set(df_viej['Año'])
    anio_pivote = max(comunes)
    
    factor = df_ref.loc[df_ref['Año'] == anio_pivote, 'Valor'].values[0] / \
             df_viej.loc[df_viej['Año'] == anio_pivote, 'Valor'].values[0]
    
    df_viej['Valor'] = df_viej['Valor'] * factor
    series_ajustadas[i] = df_viej

# --- 3. SOLUCIÓN DEFINITIVA AL EMPALME DF3 -> DF4 (Sin traslape) ---
df_ref_4 = series_ajustadas[4].sort_values('Año') # Empieza en 1950
df_viej_3 = df3_long.copy().sort_values('Año')    # Termina en 1949

# A. Obtenemos el valor de 1950 en la serie nueva (Base 2015)
val_1950_nuevo = df_ref_4.loc[df_ref_4['Año'] == 1950, 'Valor'].values[0]

# B. Calculamos el crecimiento (Gt) del último año de la serie vieja (1948 a 1949)
# Esto es para seguir la inercia de la serie antes del salto
v_1949_viej = df_viej_3.loc[df_viej_3['Año'] == 1949, 'Valor'].values[0]
v_1948_viej = df_viej_3.loc[df_viej_3['Año'] == 1948, 'Valor'].values[0]
gt_1949 = (v_1949_viej / v_1948_viej) - 1

# C. Aplicamos tu ecuación: Y(t-1) = Y(t) / (1 + Gt)
# Queremos que 1949 en base 2015 sea el valor de 1950 dividido por el crecimiento
val_1949_ajustado = val_1950_nuevo / (1 + gt_1949)

# D. Calculamos el factor de escala para todo el df3
factor_final_df3 = val_1949_ajustado / v_1949_viej

df_viej_3['Valor'] = df_viej_3['Valor'] * factor_final_df3
series_ajustadas[3] = df_viej_3

print(f"Empalme DF3 exitoso usando tasa de crecimiento Gt de 1949 ({gt_1949:.4f})")

# --- 4. UNIÓN FINAL ---
panel_final = []
# Unimos desde 8 hasta 3
for i in range(8, 2, -1):
    df_temp = series_ajustadas[i]
    if not panel_final:
        panel_final.append(df_temp)
    else:
        # Solo años que no estén en la serie superior
        anio_min_superior = pd.concat(panel_final)['Año'].min()
        panel_final.append(df_temp[df_temp['Año'] < anio_min_superior])

df_100_final = pd.concat(panel_final).sort_values('Año').reset_index(drop=True)


Empalme DF3 exitoso usando tasa de crecimiento Gt de 1949 (0.0648)


### Nota Metodológica: Empalme de la Serie Histórica (1949–1950)

La construcción de la serie unificada del Sector Agropecuario (1925–2025) presentó un desafío técnico debido al descalce entre el df3 (Base 1950) y el df4 (Base 1958). Dado que el primer bloque finaliza en 1949 y el segundo inicia en 1950, no existe un año de traslape que permita calcular un factor de empalme directo mediante métodos convencionales.

Para garantizar la continuidad estadística y evitar quiebres estructurales artificiales, se implementó un procedimiento de empalme basado en la preservación de la dinámica de crecimiento observada en la serie histórica.

**Estimación de la tasa de crecimiento:**
Se calculó la tasa de crecimiento del último intervalo disponible en la serie antigua (1948–1949):

$$
G_{1949} = \frac{Y_{1949}}{Y_{1948}} - 1
$$

Esta tasa se interpreta como una aproximación de la inercia reciente del sector y se utiliza como supuesto de continuidad para enlazar con el primer dato disponible de la nueva serie.

**Reconstrucción del nivel de enlace:**
A partir del valor observado en 1950 de la serie nueva (expresado en precios constantes de 2015), se estimó el nivel consistente para 1949 bajo el supuesto de continuidad de la tasa de crecimiento:

$$
Y_{1949}^{adj} = \frac{Y_{1950}^{Base\ 2015}}{1 + G_{1949}}
$$

**Cálculo del factor de empalme:**
Se definió un factor de escala único como el cociente entre el valor ajustado y el valor original de 1949 en la serie antigua:

$$
\lambda = \frac{Y_{1949}^{adj}}{Y_{1949}^{vieja}}
$$

**Reescalamiento de la serie histórica:**
Este factor se aplicó de manera uniforme a toda la serie comprendida entre 1925 y 1949:

$$
Y_t^{ajustada} = \lambda \cdot Y_t^{vieja}, \quad \forall t \leq 1949
$$

Este procedimiento asegura la homogeneidad en las unidades de medida (precios constantes de 2015) y preserva la estructura dinámica de la serie, permitiendo su utilización en análisis de series de tiempo —incluyendo filtros de tendencia y pruebas de estacionariedad— sin introducir sesgos derivados de cambios de base.

Finalmente, cabe señalar que este método es estándar en contextos donde no existe traslape entre series estadísticas y es consistente con prácticas empleadas en la construcción de series largas por organismos estadísticos internacionales.


# Descargar la base

Ahora vamos a descargar la base de datos que acabo de construir para trabajar todo el proceso de econometria en R.

In [49]:
# Vamos a descargar la base de datos final para trabajar en R
df_100_final.to_csv('Base_100_Anios_Construccion.csv', index=False)